# Task results

Cross-run gather tables, then a per-model drill-down reading the `results.json`/`.npz` from `02_task_training`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator='pyproject.toml',
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
# Experiment slug; matches the folder name from 01_dataset_build.
from building.polychirp_training import task_dir

COLLECTION = 'task_s3_49_2_r50'
RESULTS_DIR = task_dir(COLLECTION)  # models/<collection>/<model>/size_x<mult>/results.json
print(f'gathering: {RESULTS_DIR}')
if not RESULTS_DIR.exists():
    print('  (folder missing, run 02_task_training.ipynb first)')

## Load every run in this experiment

In [ ]:
from building.polychirp_training.results import gather_results
from building.models import BASES

df = gather_results(RESULTS_DIR)
# INT8 collapse models skew the gradient color scale; INT8 tables read df_int8, float tables read df.
collapse = {b.name for b in BASES if b.quantize_collapse}
df_int8 = df[~df['build_model'].isin(collapse)] if 'build_model' in df.columns else df
print(f'{len(df)} run(s) found')
df

## Catch up pending / malformed runs

PTQ any trained model whose `results.json` is missing or invalid, then re-run the gather cell.

In [ ]:
from building.polychirp_training.results import catch_up_pending_ptq

catch_up_pending_ptq(RESULTS_DIR, COLLECTION)

## Macro metrics side-by-side

In [ ]:
metric_cols = [
    'macro_precision', 'macro_recall', 'macro_f1', 'macro_f2',
    'macro_auc_ovr', 'subset_accuracy',
]
available = [c for c in metric_cols if c in df_int8.columns]
id_cols = [c for c in ('variant', 'model', 'build_model', 'multiplier') if c in df_int8.columns]
summary = df_int8[id_cols + available].copy()
(
    summary
    .sort_values('macro_f2', ascending=False, na_position='last')
    .style
    .background_gradient(cmap='Blues', subset=available)
    .format({c: '{:.3f}' for c in available} | {'multiplier': '{:.2f}'}, na_rep='-')
    .set_caption(f'macro metrics sorted by macro-F2: {COLLECTION}')
)

## Macro metrics: float (pre-quantization) baseline

In [ ]:
float_cols = [f'float_macro_{m}' for m in ('precision', 'recall', 'f1', 'f2', 'auc')]
available_f = [c for c in float_cols if c in df.columns]
summary_f = df[id_cols + available_f].copy()
(
    summary_f
    .sort_values('float_macro_f2', ascending=False, na_position='last')
    .style
    .background_gradient(cmap='Blues', subset=available_f)
    .format({c: '{:.3f}' for c in available_f} | {'multiplier': '{:.2f}'}, na_rep='-')
    .set_caption(f'macro metrics (float baseline) sorted by macro-F2: {COLLECTION}')
)

## Per-class F2: hardest classes

In [ ]:
from building.polychirp_training.results import per_class_metric_table
import pandas as pd

f2_pc = per_class_metric_table(df_int8, 'f2')
if f2_pc.empty:
    print('no per-class F2 data')
else:
    w = df_int8.set_index(['variant', 'model'])['macro_f2'].reindex(f2_pc.index).fillna(0.0)
    w = w.to_numpy()[:, None]
    mask = f2_pc.notna().to_numpy()
    num = (f2_pc.fillna(0.0).to_numpy() * w * mask).sum(axis=0)
    den = (w * mask).sum(axis=0)
    weighted = pd.Series(num / den, index=f2_pc.columns, name='weighted_F2')
    weighted = weighted.sort_values()
    display(
        weighted.to_frame().style
        .background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=1)
        .format('{:.3f}', na_rep='-')
        .set_caption(f'per-class F2 weighted by model macro-F2: {COLLECTION}')
    )

## Quantization drop

In [ ]:
# INT8 minus float per macro metric: negative = drop after quantization.
metrics = ['precision', 'recall', 'f1', 'f2', 'auc']
drops = pd.DataFrame({'model': df_int8['model']})
for m in metrics:
    f, q = f'float_macro_{m}', f'macro_{m}'
    if f in df_int8.columns and q in df_int8.columns:
        drops[m] = df_int8[q] - df_int8[f]
drops = drops.set_index('model')
sort_col = 'f2' if 'f2' in drops.columns else next(iter(drops.columns), None)
if sort_col is None:
    print('no standard runs with a float baseline, nothing to compare')
else:
    drops = drops.sort_values(sort_col, na_position='last')
    cols = list(drops.columns)
    display(
        drops.style
        .background_gradient(cmap='RdYlGn', subset=cols, axis=None, vmin=-0.05, vmax=0.05)
        .format({c: '{:+.3f}' for c in cols}, na_rep='-')
        .set_caption(f'quantization drop (INT8 - float): {COLLECTION}')
    )

# Per-model drill-down

Pick one model (`build_model` + `multiplier`) from the gather table; everything below reads its `results.json`/`.npz`.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from building.polychirp_training import run_paths
from building.polychirp_training.results import load_run
from building.models import model_eval as M

MODEL_NAME = 'sincnet'   # a `build_model` value from the gather table above
MULTIPLIER = 1           # the matching `multiplier`
SPLIT = 'test'           # 'train' (= train+val) | 'test'
OBJ = 'f2'               # threshold objective: 'f1' | 'f2'

paths = run_paths(COLLECTION, MODEL_NAME, MULTIPLIER)
loaded = load_run(paths.results_path)
assert loaded is not None, (
    f'no results.json at {paths.results_path}, train it (02_task_training) '
    'or run the catch-up cell above'
)
record = loaded.record
cmp = loaded.cmp(SPLIT, OBJ)
float_eval = loaded.eval('keras', SPLIT, OBJ)
quant_eval = loaded.eval('tflite', SPLIT, OBJ)
label_names = loaded.label_names
print(f'{MODEL_NAME} x{MULTIPLIER:g}  split={SPLIT}  thr-opt={OBJ.upper()}')
print(f'labels: {label_names}')
M.print_summary(quant_eval)

## Per-species metrics (quantized)

In [ ]:
tbl = M.per_class_table(quant_eval)
display(tbl.style.format(precision=4).set_caption('Quantized per-species'))
print(
    f'Macro: precision={quant_eval.macro_precision:.4f}  '
    f'recall={quant_eval.macro_recall:.4f}  '
    f'F1={quant_eval.macro_f1:.4f}  '
    f'F2={quant_eval.macro_f2:.4f}  '
    f'AUC={quant_eval.macro_auc:.4f}'
)

## Metric-vs-threshold sweep (paper Fig. 5 style, quantized)

In [ ]:
fig = M.plot_metric_sweep_panel(quant_eval)
plt.show()

## ROC per species (quantized) and AUC

In [ ]:
fig = M.plot_roc(quant_eval)
plt.show()
for name, mclass in quant_eval.per_class.items():
    print(f'  {name:<28s} AUC={mclass.auc:.4f}' if mclass.auc is not None else f'  {name:<28s} n/a')

## Float vs INT8 comparison

In [ ]:
display(M.comparison_table(cmp).style.set_caption('Float vs INT8 per-class'))
fig = M.plot_quantization_drop(cmp, metric='f2')
plt.show()
M.print_comparison(cmp)

## Prediction rates by true class (quantized)

Diagonal = recall; rows can exceed 1 (independent multi-label sigmoid).

In [ ]:
M.plot_prediction_rates(quant_eval)
plt.show()

## Per-model measurement tables

(float | INT8) x (train | test) x (F1 | F2 threshold), tuned on train+val per backend.

In [ ]:
M.display_opti_tables(record)

### Confusion matrices (top-1)

In [ ]:
M.plot_confusion_matrices(record.confusion, label_names)
plt.show()